# Week 3 Day 2 — Fitting the Svensson Curve

Tests for `fit_svensson` in `src/termstructure/curves/svensson.py`.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from termstructure.curves.svensson import svensson_zero_rate, fit_svensson

## 1. Load a single date

We'll fit to the 30 zero yields published by the Fed for 2020-01-15.

In [ ]:
df = pd.read_parquet('../data/processed/treasury_bonds.parquet')
df['date'] = pd.to_datetime(df['date'])

row = df[df['date'] == '2020-01-15'].iloc[0]

maturities = np.arange(1, 31, dtype=float)
yields = np.array([row[f'sveny{i:02d}'] for i in range(1, 31)])

print(f"Date: {row['date'].date()}")
print(f"Yields (1Y to 30Y): {yields[:5].round(4)} ... {yields[-3:].round(4)}")

## 2. Fit and check RMSE

A good fit should be under 1 bp RMSE. Under 0.5 bp is excellent.

In [ ]:
params = fit_svensson(maturities, yields)
b0, b1, b2, b3, l1, l2 = params

fitted = svensson_zero_rate(maturities, *params)
residuals_bp = (fitted - yields) * 100
rmse_bp = np.sqrt(np.mean(residuals_bp**2))

print(f"Our fit:   b0={b0:.4f}  b1={b1:.4f}  b2={b2:.4f}  b3={b3:.4f}")
print(f"           l1={l1:.4f}  l2={l2:.4f}")
print()
print(f"Fed params: b0={row.beta0:.4f}  b1={row.beta1:.4f}  b2={row.beta2:.4f}  b3={row.beta3:.4f}")
print(f"            l1={row.tau1:.4f}  l2={row.tau2:.4f}")
print()
print(f"RMSE: {rmse_bp:.3f} bp")

**Why our parameters differ from the Fed's**

Svensson is an ill-conditioned problem — many different parameter combinations can produce curves that fit the data equally well. Think of it like a regression where two predictors are highly correlated: the individual coefficients vary wildly but the predictions are stable. What matters is the RMSE (curve fit quality), not the individual parameter values. Both our fit and the Fed's describe essentially the same curve.

## 3. Plot fitted curve vs. observed yields

In [ ]:
tau_fine = np.linspace(0.5, 30, 300)
our_curve = svensson_zero_rate(tau_fine, *params)
fed_curve = svensson_zero_rate(tau_fine, row.beta0, row.beta1, row.beta2, row.beta3, row.tau1, row.tau2)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), gridspec_kw={'height_ratios': [3, 1]})

ax1.plot(tau_fine, our_curve, label='Our fit', color='steelblue')
ax1.plot(tau_fine, fed_curve, label="Fed's fit", color='tomato', linestyle='--')
ax1.scatter(maturities, yields, label='Observed (sveny)', color='black', s=20, zorder=5)
ax1.set_ylabel('Zero yield (%)')
ax1.set_title('Svensson fit — 2020-01-15')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.bar(maturities, residuals_bp, color='steelblue', alpha=0.7)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_xlabel('Maturity (years)')
ax2.set_ylabel('Residual (bp)')
ax2.set_title(f'Fit residuals  (RMSE = {rmse_bp:.3f} bp)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Test on a stress-period date (March 2020)

COVID crash — the curve was extremely dislocated. A robust fitter should still converge.

In [ ]:
row2 = df[df['date'] == '2020-03-18'].iloc[0]
yields2 = np.array([row2[f'sveny{i:02d}'] for i in range(1, 31)])

params2 = fit_svensson(maturities, yields2)
fitted2 = svensson_zero_rate(maturities, *params2)
rmse2 = np.sqrt(np.mean(((fitted2 - yields2) * 100)**2))

plt.figure(figsize=(9, 4))
plt.plot(np.linspace(0.5, 30, 300),
         svensson_zero_rate(np.linspace(0.5, 30, 300), *params2),
         label='Fitted', color='steelblue')
plt.scatter(maturities, yields2, color='black', s=20, zorder=5, label='Observed')
plt.xlabel('Maturity (years)')
plt.ylabel('Zero yield (%)')
plt.title(f'Svensson fit — 2020-03-18 (COVID crash)  RMSE={rmse2:.3f} bp')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"RMSE: {rmse2:.3f} bp")

## 5. Sensitivity to starting guess

Svensson is nonconvex — different starting points can give different parameter values.
Check that RMSE is stable even when we perturb the starting guess.

In [ ]:
from scipy.optimize import least_squares

rng = np.random.default_rng(42)
weights = 1.0 / maturities

def residuals_fn(p):
    return weights * (svensson_zero_rate(maturities, *p) - yields)

bounds = ([-np.inf]*4 + [0.1, 0.1], [np.inf]*4 + [10., 10.])
rmses = []

for _ in range(20):
    x0 = [yields[-1] + rng.normal(0, 0.2),
          yields[0] - yields[-1] + rng.normal(0, 0.2),
          rng.normal(0, 0.1),
          rng.normal(0, 0.1),
          rng.uniform(0.5, 5),
          rng.uniform(0.5, 8)]
    res = least_squares(residuals_fn, x0, method='trf', bounds=bounds)
    rmse = np.sqrt(np.mean((res.fun / weights)**2)) * 100
    rmses.append(rmse)

print(f"RMSE across 20 random starts: min={min(rmses):.3f}  max={max(rmses):.3f}  mean={np.mean(rmses):.3f} bp")
print("Parameters vary but curve quality is stable — this is expected for Svensson.")